# Day 3 – Exercise Solutions

Solutions for tokenisation and embeddings exercises.

## Exercise 1: Token cost estimator

In [ ]:
import tiktoken

def estimate_cost(filepath, price_per_1k=0.0015):
    enc = tiktoken.get_encoding("cl100k_base")
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    tokens = enc.encode(text)
    num_tokens = len(tokens)
    cost = (num_tokens / 1000) * price_per_1k
    print(f"Tokens: {num_tokens}")
    print(f"Estimated cost: ${cost:.4f}")
    return num_tokens, cost

# Example (uncomment and provide a real file path)
# estimate_cost("chapter1.txt")

## Exercise 2: Find the odd one out

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    "I love pizza.",
    "Pasta is delicious.",
    "The weather is nice today.",
    "Spaghetti with meatballs is great."
]

embeddings = model.encode(sentences)
similarity_matrix = cosine_similarity(embeddings)

# For each sentence, compute average similarity to all others
avg_similarities = [np.mean(similarity_matrix[i, :i] + similarity_matrix[i, i+1:]) for i in range(len(sentences))]
odd_index = np.argmin(avg_similarities)
print(f"Odd one out: '{sentences[odd_index]}' with avg similarity {avg_similarities[odd_index]:.3f}")

## Exercise 3: Visualise embeddings with PCA

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Using the same sentences and embeddings as above
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

plt.figure(figsize=(8,6))
for i, sentence in enumerate(sentences):
    plt.scatter(embeddings_2d[i,0], embeddings_2d[i,1])
    plt.annotate(sentence[:20], (embeddings_2d[i,0], embeddings_2d[i,1]))
plt.title("2D PCA of Sentence Embeddings")
plt.show()

## Exercise 4: Multilingual tokenisation comparison

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

texts = {
    "English": "The cat sat on the mat.",
    "Spanish": "El gato se sentó en la alfombra.",
    "Japanese": "猫がマットの上に座った。"
}

for lang, text in texts.items():
    tokens = enc.encode(text)
    print(f"{lang:10} -> {len(tokens):2} tokens")
# Japanese typically uses more tokens per character because of Unicode encoding.

## Exercise 5: Embedding search engine

In [ ]:
def search_engine(corpus, query, model, top_k=3):
    corpus_emb = model.encode(corpus)
    query_emb = model.encode([query])
    similarities = cosine_similarity(query_emb, corpus_emb)[0]
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    for idx in top_indices:
        print(f"Score {similarities[idx]:.3f}: {corpus[idx]}")

# Example
corpus = [
    "Python is a programming language.",
    "Jupyter notebooks are great for data science.",
    "Embeddings represent meaning as vectors.",
    "The weather today is sunny.",
    "Tokenization splits text into smaller pieces."
]
query = "How do we represent text for machine learning?"
search_engine(corpus, query, model)